In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import anndata as ad

In [ ]:
import yaml

base_path = Path('../..').resolve()
sys.path.append(str(base_path))
from helpers import singlecell_utils

with open(base_path / 'config.yaml') as f:
    cfg = yaml.safe_load(f)

In [ ]:
# Need Big memory

FULL_H5AD_PATH = '/sc/arion/projects/psychAD/NPS-AD/freeze2_proc/240124_PsychAD_freeze3_FULL_clean.h5ad'
psyad_full = singlecell_utils.read_adata_withraw_except_x(FULL_H5AD_PATH)

In [ ]:
psyad_full.var.head()

# Class level

In [ ]:
df_psb = singlecell_utils.get_pseudobulk_from_raw(psyad_full, 'class')
df_psb.head()

In [ ]:
df_psb.max()

In [ ]:
df_psb_cpm = (df_psb+0.5) / ((df_psb+0.5).sum() / 10**6)

for col in df_psb:
    df_psb['cpm_' + col] = df_psb_cpm[col]
    
df_psb

In [ ]:
# PSB and var has the same index

(df_psb.index == psyad_full.var.index).all()

In [ ]:
df_psb.to_pickle('PsychAD_class_psb_cnt_cpm.pkl')
df_psb.to_csv('PsychAD_class_psb_cnt_cpm.tsv.gz', sep='\t', compression='gzip')

# Subclass level

In [ ]:
df_psb = singlecell_utils.get_pseudobulk_from_raw(psyad_full, 'subclass')
df_psb.head()

In [ ]:
df_psb.max()

In [ ]:
df_psb_cpm = (df_psb+0.5) / ((df_psb+0.5).sum() / 10**6)

for col in df_psb:
    df_psb['cpm_' + col] = df_psb_cpm[col]
    
df_psb

In [ ]:
# PSB and var has the same index

(df_psb.index == psyad_full.var.index).all()

In [ ]:
df_psb.to_pickle('PsychAD_subclass_psb_cnt_cpm.pkl')
df_psb.to_csv('PsychAD_subclass_psb_cnt_cpm.tsv.gz', sep='\t', compression='gzip')

# Age X class level

In [ ]:
AGE_H5AD_PATH = '/sc/arion/projects/psychAD/NPS-AD/freeze2_rc/h5ad_final/AGING_2024-02-01_22_23.h5ad'
dat_age = singlecell_utils.read_everything_but_X(AGE_H5AD_PATH)

In [ ]:
pd.crosstab(dat_age.obs.r10x, dat_age.obs['class'])

In [ ]:
# Each SubID has unique age
dat_age.obs.groupby('SubID', observed=True).r10x.nunique().max()

In [ ]:
se_ages = dat_age.obs.groupby('SubID', observed=True).r10x.first()
subid_to_age = dict(zip(se_ages.index, se_ages))

In [ ]:
psyad_full.obs['r10x'] = psyad_full.obs.SubID.map(subid_to_age)

In [ ]:
psyad_full.obs['age_X_class'] = psyad_full.obs['r10x'].astype(str) + '-' + psyad_full.obs['class'].astype(str) 

In [ ]:
df_psb = singlecell_utils.get_pseudobulk_from_raw(psyad_full, 'age_X_class')
df_psb.head()

In [ ]:
df_psb.max()

In [ ]:
df_psb_cpm = (df_psb+0.5) / ((df_psb+0.5).sum() / 10**6)

for col in df_psb:
    df_psb['cpm_' + col] = df_psb_cpm[col]
    
df_psb

In [ ]:
df_psb.sum().tail(20)

In [ ]:
# PSB and var has the same index

(df_psb.index == psyad_full.var.index).all()

In [ ]:
df_psb.to_pickle('PsychAD_age_X_class_psb_cnt_cpm.pkl')
df_psb.to_csv('PsychAD_age_X_class_psb_cnt_cpm.tsv.gz', sep='\t', compression='gzip')